Binary Classification - Nostalgia/Not nostalgia (sentiment analysis)
Dataset: https://huggingface.co/datasets/Senem/Nostalgic_Sentiment_Analysis_of_YouTube_Comments_Data

Models:

google-bert/bert-base-cased : https://huggingface.co/google-bert/bert-base-cased

FacebookAI/xlm-roberta-base: https://huggingface.co/FacebookAI/xlm-roberta-base






In [ ]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install accelerate --upgrade

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from torch.utils.tensorboard import SummaryWriter
import re
import torch
import tensorflow as tf
import tensorboard as tb

In [ ]:
import numpy as np
import evaluate
from transformers import AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding, AutoModelForSequenceClassification
from datasets import load_dataset, DatasetDict


In [ ]:
# Just take the first 100 tokens for speed
label_map = {
    "not nostalgia": 0,
    "nostalgia": 1
}
def truncate(example):
    return {
        'comment': " ".join(example['comment'].split()), #<- removed limitation in truncate function to get whole sequence
        'sentiment': label_map[example['sentiment']]
    }

nostalgia_dataset = load_dataset("Senem/Nostalgic_Sentiment_Analysis_of_YouTube_Comments_Data")

#take 256 random examples for train and 64 validation & 64 for testing
small_nostalgia_dataset = DatasetDict(
    train= nostalgia_dataset['train'].shuffle(seed=224).select(range(512)).map(truncate),
    val=nostalgia_dataset['train'].shuffle(seed=224).select(range(512, 576)).map(truncate),
    test=nostalgia_dataset['train'].shuffle(seed=224).select(range(576, 640)).map(truncate), #<-- this is the first version/ i also tried increasing the test set by 512 made the evaluation metrics go down eventually
)


Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

In [ ]:
small_nostalgia_dataset

DatasetDict({
    train: Dataset({
        features: ['sentiment', 'comment'],
        num_rows: 512
    })
    val: Dataset({
        features: ['sentiment', 'comment'],
        num_rows: 64
    })
    test: Dataset({
        features: ['sentiment', 'comment'],
        num_rows: 64
    })
})

**FacebookAI**

In [ ]:
#facebookAI
tokenizer_fb = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
data_collator_fb = DataCollatorWithPadding(tokenizer=tokenizer_fb)

def tokenize_function_fb(examples):
    tokenized = tokenizer_fb(examples["comment"], padding=True, truncation=True)
    tokenized["labels"] = examples["sentiment"]
    return tokenized

print(small_nostalgia_dataset['train'].column_names)


small_tokenized_dataset_fb = small_nostalgia_dataset.map(tokenize_function_fb, batched=True, batch_size=16)
data_collator_fb = DataCollatorWithPadding(tokenizer=tokenizer_fb)

['sentiment', 'comment']


Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

In [ ]:
#training FacebookAI
model_fb = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
accuracy = evaluate.load("accuracy")

arguments = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/cl_facebookai",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to='none',
    seed=224
)

def compute_metrics(eval_pred):
    """Called at the end of validation. Gives accuracy"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)


trainer = Trainer(
    model=model_fb,
    args=arguments,
    train_dataset=small_tokenized_dataset_fb['train'],
    eval_dataset=small_tokenized_dataset_fb['val'],
    processing_class=tokenizer_fb,
    data_collator=data_collator_fb,
    compute_metrics=compute_metrics
)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.435600,0.286404,0.890625
2,0.317400,0.371357,0.875000
3,0.240000,0.122410,0.968750
4,0.171500,0.132252,0.968750
5,0.128100,0.144523,0.968750


TrainOutput(global_step=160, training_loss=0.2723478427156806, metrics={'train_runtime': 446.552, 'train_samples_per_second': 5.733, 'train_steps_per_second': 0.358, 'total_flos': 161583076872960.0, 'train_loss': 0.2723478427156806, 'epoch': 5.0})

In [ ]:
#visualisation f facebookai
fine_tuned_fb_model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/cl_facebookai/checkpoint-320")

model_inputs = tokenizer_fb(list(small_tokenized_dataset_fb['test']['comment']), padding=True, truncation=True, return_tensors='pt')
outputs_fb = fine_tuned_fb_model(**model_inputs, output_hidden_states=True) #use test set!!

In [ ]:
import os


path = "/content/drive/MyDrive/Colab Notebooks/cl_facebookai/results_vis_fb1.1"
layer=0
if not os.path.exists(path):
  os.mkdir(path)

while layer in range(len(outputs_fb['hidden_states'])):
  if not os.path.exists(path+'/layer_' + str(layer)):
    os.mkdir(path+'/layer_' + str(layer))

  example = 0
  tensors = []
  labels = []

  while example in range(len(outputs_fb['hidden_states'][layer])):
    sp_token_position = 0
    for token in model_inputs['input_ids'][example]:
      if token != 0:
        sp_token_position += 1
      else:
        tensor = outputs_fb['hidden_states'][layer][example][sp_token_position]
        tensors.append(tensor)
        break

    label = [small_tokenized_dataset_fb['test']['comment'][example],str(small_tokenized_dataset_fb['test']['sentiment'][example])] #test dataset!!
    labels.append(label)
    example +=1

  writer=SummaryWriter(path+'/layer_' + str(layer))
  writer.add_embedding(torch.stack(tensors), metadata=labels, metadata_header=['Text','Emotion'])

  layer+=1

**Google BERT**

In [ ]:
#google bert

tokenizer_google = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
def tokenize_function_google(examples):
    tokenized = tokenizer_google(examples["comment"], padding=True, truncation=True)
    tokenized["labels"] = examples["sentiment"]
    return tokenized

small_tokenized_dataset_google = small_nostalgia_dataset.map(tokenize_function_google, batched=True, batch_size=16)

data_collator_google = DataCollatorWithPadding(tokenizer=tokenizer_google)


Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

In [ ]:
#training googlebert
model_google = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=2)
accuracy = evaluate.load("accuracy")

arguments = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/cl_google",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=8,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to='none',
    seed=224
)

def compute_metrics(eval_pred):
    """Called at the end of validation. Gives accuracy"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)


trainer = Trainer(
    model=model_google,
    args=arguments,
    train_dataset=small_tokenized_dataset_google['train'],
    eval_dataset=small_tokenized_dataset_google['val'],
    processing_class=tokenizer_google,
    data_collator=data_collator_google,
    compute_metrics=compute_metrics
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.452500,0.373251,0.859375
2,0.200500,0.251114,0.906250
3,0.103500,0.171984,0.937500
4,0.071800,0.137730,0.968750
5,0.010600,0.090680,0.968750
6,0.043300,0.093511,0.968750


TrainOutput(global_step=192, training_loss=0.16722214749703804, metrics={'train_runtime': 263.9077, 'train_samples_per_second': 11.64, 'train_steps_per_second': 0.728, 'total_flos': 197629291457280.0, 'train_loss': 0.16722214749703804, 'epoch': 6.0})

In [ ]:
#visualisation for google
fine_tuned_google_model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/cl_google/checkpoint-448")

model_inputs = tokenizer_google(list(small_tokenized_dataset_google['test']['comment']), padding=True, truncation=True, return_tensors='pt')
outputs_google = fine_tuned_google_model(**model_inputs, output_hidden_states=True)

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/cl_google/results_vis_google1.1"
layer=0
if not os.path.exists(path):
  os.mkdir(path)

while layer in range(len(outputs_google['hidden_states'])):
  if not os.path.exists(path+'/layer_' + str(layer)):
    os.mkdir(path+'/layer_' + str(layer))

  example = 0
  tensors = []
  labels = []

  while example in range(len(outputs_google['hidden_states'][layer])):
    sp_token_position = 0
    for token in model_inputs['input_ids'][example]:
      if token != 101:
        sp_token_position += 1
      else:
        tensor = outputs_google['hidden_states'][layer][example][sp_token_position]
        tensors.append(tensor)
        break

    label = [small_tokenized_dataset_google['test']['comment'][example],str(small_tokenized_dataset_google['test']['sentiment'][example])]
    labels.append(label)
    example +=1

  writer=SummaryWriter(path+'/layer_' + str(layer))
  writer.add_embedding(torch.stack(tensors), metadata=labels, metadata_header=['Text','Emotion'])

  layer+=1

**Evaluation on Test Set**

In [ ]:
import evaluate
from transformers import AutoModelForSequenceClassification
import torch

metric = evaluate.load("accuracy")
fine_tuned_model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/cl_facebookai/checkpoint-320")

model_inputs = tokenizer_fb(list(small_tokenized_dataset_fb['test']['comment']), padding=True, truncation=True, return_tensors='pt')
outputs = fine_tuned_model(**model_inputs, output_hidden_states=True)
predictions = torch.argmax(outputs.logits, dim=-1)
accuracy = metric.compute(predictions=predictions, references=small_tokenized_dataset_fb['test']['sentiment'])
print(accuracy)

{'accuracy': 1.0}


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef

model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/cl_facebookai/checkpoint-320")
model.eval()
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

test_comments = list(small_tokenized_dataset_fb['test']['comment'])
true_labels = list(small_tokenized_dataset_fb['test']['sentiment'])

inputs = tokenizer(test_comments, padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1).tolist()

acc = accuracy_score(true_labels, predictions)
prec = precision_score(true_labels, predictions)
rec = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)
mcc = matthews_corrcoef(true_labels, predictions)

print("FacebookAI Model Results:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"MCC:       {mcc:.4f}")

FacebookAI Model Results:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
MCC:       1.0000


In [ ]:
model_google = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/cl_google/checkpoint-448")
model_google.eval()
tokenizer_google = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")


test_comments_google = list(small_tokenized_dataset_google['test']['comment'])
true_labels_google = list(small_tokenized_dataset_google['test']['sentiment'])


inputs_google = tokenizer_google(test_comments_google, padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    outputs_google = model_google(**inputs_google)
    predictions_google = torch.argmax(outputs_google.logits, dim=-1).tolist()

acc_google = accuracy_score(true_labels_google, predictions_google)
prec_google = precision_score(true_labels_google, predictions_google)
rec_google = recall_score(true_labels_google, predictions_google)
f1_google = f1_score(true_labels_google, predictions_google)
mcc_google = matthews_corrcoef(true_labels_google, predictions_google)

print("Google BERT Model Results:")
print(f"Accuracy:  {acc_google:.4f}")
print(f"Precision: {prec_google:.4f}")
print(f"Recall:    {rec_google:.4f}")
print(f"F1-Score:  {f1_google:.4f}")
print(f"MCC:       {mcc_google:.4f}")

Google BERT Model Results:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
MCC:       1.0000


**Examining the dataset a bit further**

In [ ]:
test_str = "I remember my father used to listen to Jim Reeves when I was young. I am now 64." #classified as not nostalgic in dataset
#I was in 2nd grade when this movie came out. When you age, your memory is your only treasure. Build a treasure for your youth. <-- classified as nostalgic by model/labelled as not nostalgic
fine_tuned_model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/Colab Notebooks/cl_facebookai/checkpoint-320")
model_inputs = tokenizer(test_str, return_tensors="pt")
prediction = torch.argmax(fine_tuned_model(**model_inputs).logits)
print(["NOT NOSTALGIC", "NOSTALGIC"][prediction])

NOSTALGIC


In [ ]:
def find_duplicates(dataset, field="comment"):

    seen = {}
    duplicate_indices = {}

    for idx, value in enumerate(dataset[field]):
        if value in seen:
            if value not in duplicate_indices:
                duplicate_indices[value] = [seen[value]]
            duplicate_indices[value].append(idx)
        else:
            seen[value] = idx

    duplicates = list(duplicate_indices.keys())
    return duplicates, duplicate_indices

dups, dup_idx = find_duplicates(small_nostalgia_dataset["train"])

print(f"Number of duplicate comments: {len(dups)}")

if dups:
    first = dups[0]
    print("Example duplicate:", first)
    print("Indices:", dup_idx[first])
else:
    print("No duplicates found.")


Number of duplicate comments: 0
No duplicates found.


In [ ]:
#Source
#@article{postalcioglu2020comparison,
#  title={Comparison of Neural Network Models for Nostalgic Sentiment Analysis of YouTube Comments},
#  author={Postalcioglu, Seda and Aktas, Senem},
#  journal={Hittite Journal of Science and Engineering},
#  volume={7},
#  number={3},
#  pages={215--221},
#  year={2020},
#  publisher={Hitit University}
#}